In [ ]:
%pip install openpyxl

In [2]:
import pandas as pd
import numpy as np

Dataset Source - Synthetically generated dataset.

The dataset, retail_matcha_sales_messy_dataset_v4.xlsx, was created with the assistance of generative AI for the purpose of this data cleaning and preparation assignment. It represents a fictional retail matcha sales dataset containing transaction, product, pricing, payment, location and customer rating information.

The dataset was intentionally designed to contain data quality issues, including missing values, inconsistent categorical labels, duplicate records and incorrectly formatted values, to demonstrate data cleaning and preparation techniques.

Access and reproducibility:

The original Excel dataset is included in the data/ directory of this repository to ensure reproducibility.

In [3]:
#read uploaded excel file and save it into "sales" valiable
from pathlib import Path

DATA_PATH = Path("data") / "retail_matcha_sales_messy_dataset_v4.xlsx"

sales = pd.read_excel(DATA_PATH)

In [4]:
#display first 5 entries in the data set
sales.head()

,Transaction_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price_EUR,Sales_Channel,Payment_Method,Location,Order_Total_EUR,Customer_Rating,Returned,Staff_ID
0,TXN-10001,2026-01-01,CUST-0052,Ceremonial Matcha 30g,Retail,1,24,Wholesale,Cash,Wholesale,24.0,4,Yes,STF-01
1,TXN-10002,2026-01-01,CUST-0056,Classic Matcha Latte,Drinks,1,5.5,In-store,Apple Pay,Mahon Pop-up,5.5,3,No,STF-03
2,TXN-10003,2026-01-01,CUST-0028,Iced Hojicha Latte,Drinks,3,5.8,In-store,Apple Pay,Mahon Pop-up,17.4,5,No,STF-01
3,TXN-10004,2026-01-03,CUST-0072,Iced Hojicha Latte,Drinks,2,5.8,Wholesale,Card,Marina Market,11.6,3,Yes,STF-04
4,TXN-10005,2026-01-08,CUST-0055,Matcha Granola,Food,3,7.5,Online,Apple Pay,Paul Street Pop-up,22.5,NaN,No,STF-01


In [5]:
print("Number of rows:", sales.shape[0]) #number of rows
print("Number of columns:", sales.shape[1]) #number of columns
print("\n") #new blank row for better visibility

#displayes names of all the columns in the data set
sales.columns

Number of rows: 255
Number of columns: 14




Index(['Transaction_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category',
       'Quantity', 'Unit_Price_EUR', 'Sales_Channel', 'Payment_Method',
       'Location', 'Order_Total_EUR', 'Customer_Rating', 'Returned',
       'Staff_ID'],
      dtype='object')

In [6]:
#displays complete summary of the data set, we can also see all 14 column names and different number of rows for some columns
# which points out to the fact that there are some missing values in these columns
#all data types are currently "object" but Order_Total_Eur which is "float"

sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Transaction_ID   255 non-null    object 
 1   Order_Date       249 non-null    object 
 2   Customer_ID      242 non-null    object 
 3   Product          255 non-null    object 
 4   Category         255 non-null    object 
 5   Quantity         255 non-null    object 
 6   Unit_Price_EUR   255 non-null    object 
 7   Sales_Channel    249 non-null    object 
 8   Payment_Method   251 non-null    object 
 9   Location         251 non-null    object 
 10  Order_Total_EUR  255 non-null    float64
 11  Customer_Rating  161 non-null    object 
 12  Returned         255 non-null    object 
 13  Staff_ID         204 non-null    object 
dtypes: float64(1), object(13)
memory usage: 28.0+ KB


In [7]:
#checking for number of unique values/cathegories in each column
unique = sales.nunique()
print(unique)

Transaction_ID     250
Order_Date         155
Customer_ID         85
Product              8
Category             6
Quantity            10
Unit_Price_EUR      12
Sales_Channel        8
Payment_Method       8
Location             9
Order_Total_EUR     41
Customer_Rating      7
Returned             7
Staff_ID             4
dtype: int64


In [8]:
#detects and counts missing values for each column and then prints them
missing = sales.isna().sum()
print(missing, "\n")

#displys the percentage of missing values and round to 1 decimal point
print((sales.isna().mean()*100).round(1))


Transaction_ID      0
Order_Date          6
Customer_ID        13
Product             0
Category            0
Quantity            0
Unit_Price_EUR      0
Sales_Channel       6
Payment_Method      4
Location            4
Order_Total_EUR     0
Customer_Rating    94
Returned            0
Staff_ID           51
dtype: int64 

Transaction_ID      0.0
Order_Date          2.4
Customer_ID         5.1
Product             0.0
Category            0.0
Quantity            0.0
Unit_Price_EUR      0.0
Sales_Channel       2.4
Payment_Method      1.6
Location            1.6
Order_Total_EUR     0.0
Customer_Rating    36.9
Returned            0.0
Staff_ID           20.0
dtype: float64


We have missing values in following columns:

Knowing the data set, Transaction ID and Order Date are both in ascending order so we could estimate the missing dates
*   Order Date

We cannot safely estimate missing values in these columns so we change text to "Unknown"
*   Customer_ID
*   Sales_Channel
*   Payment_Method
*   Location
*   Customer_Rating
*   Staff_ID


In [9]:
#display Transaction_ID rows with missing Order_Date
sales.loc[
    sales["Order_Date"].isna(),
    ["Transaction_ID", "Order_Date"]
]

,Transaction_ID,Order_Date
81,TXN-10080,NaN
82,TXN-10081,NaN
83,TXN-10082,NaN
182,TXN-10180,NaN
183,TXN-10181,NaN
184,TXN-10182,NaN


In [10]:
#displaying the rows with missing dates and rows around to verify the continuity
for start, end in [(78, 86), (179, 187)]:
    print(sales.loc[
        start:end,
        ["Transaction_ID", "Order_Date"]
    ])
    print("-" * 40)

   Transaction_ID  Order_Date
78      TXN-10077  2026-04-02
79      TXN-10078  2026-04-05
80      TXN-10079  2026-04-05
81      TXN-10080         NaN
82      TXN-10081         NaN
83      TXN-10082         NaN
84      TXN-10083  2026-04-07
85      TXN-10084  2026-04-08
86      TXN-10085  2026-04-14
----------------------------------------
    Transaction_ID  Order_Date
179      TXN-10177  2026-08-11
180      TXN-10178  2026-08-11
181      TXN-10179  2026-08-11
182      TXN-10180         NaN
183      TXN-10181         NaN
184      TXN-10182         NaN
185      TXN-10183  2026-08-16
186      TXN-10184  2026-08-17
187      TXN-10185  2026-08-20
----------------------------------------


In [11]:
#using forward fill method to assign missing dates to correct period
sales["Order_Date"] = sales["Order_Date"].ffill()


The forward fill method was selected as a simple imputation technique that preserves the chronological order of the dataset without introducing estimated dates. However, we assume that the missing order dates may correspond to the dates of preceding transactions. As the original dates could not be verified, the imputed values should be taken as estimates rather than confirmed dates.

In [12]:
#checking the result
for start, end in [(78, 86), (179, 187)]:
    print(sales.loc[
        start:end,
        ["Transaction_ID", "Order_Date"]
    ])
    print("-" * 40)

   Transaction_ID  Order_Date
78      TXN-10077  2026-04-02
79      TXN-10078  2026-04-05
80      TXN-10079  2026-04-05
81      TXN-10080  2026-04-05
82      TXN-10081  2026-04-05
83      TXN-10082  2026-04-05
84      TXN-10083  2026-04-07
85      TXN-10084  2026-04-08
86      TXN-10085  2026-04-14
----------------------------------------
    Transaction_ID  Order_Date
179      TXN-10177  2026-08-11
180      TXN-10178  2026-08-11
181      TXN-10179  2026-08-11
182      TXN-10180  2026-08-11
183      TXN-10181  2026-08-11
184      TXN-10182  2026-08-11
185      TXN-10183  2026-08-16
186      TXN-10184  2026-08-17
187      TXN-10185  2026-08-20
----------------------------------------


In [13]:
#filling the rest of NaN with "Unknown"
sales = sales.fillna("Unknown")

In [14]:
#checking if any NaN values are left in data set
sales.isna().sum()

Transaction_ID     0
Order_Date         0
Customer_ID        0
Product            0
Category           0
Quantity           0
Unit_Price_EUR     0
Sales_Channel      0
Payment_Method     0
Location           0
Order_Total_EUR    0
Customer_Rating    0
Returned           0
Staff_ID           0
dtype: int64

In [15]:
#Find number of duplicate rows

duplicates = sales.duplicated()
print("Number of duplicate rows:", duplicates.sum())
print('\n')

#list Duplicite orders - Transaction IDs:
for row in sales.index:
    if duplicates[row]:
        print("Duplicate order - Transaction ID:", sales.loc[row, "Transaction_ID"])


Number of duplicate rows: 5


Duplicate order - Transaction ID: TXN-10018
Duplicate order - Transaction ID: TXN-10069
Duplicate order - Transaction ID: TXN-10144
Duplicate order - Transaction ID: TXN-10202
Duplicate order - Transaction ID: TXN-10230


In [16]:
#checking if duplicate orders are really completely same - listing all instances:
sales[sales.duplicated(keep=False)]

,Transaction_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price_EUR,Sales_Channel,Payment_Method,Location,Order_Total_EUR,Customer_Rating,Returned,Staff_ID
17,TXN-10018,2026-01-24,CUST-0002,Pistachio Matcha,Drinks,3,6.5,Pop-up,Online,Paul Street Pop-up,19.5,4,Yes,STF-04
18,TXN-10018,2026-01-24,CUST-0002,Pistachio Matcha,Drinks,3,6.5,Pop-up,Online,Paul Street Pop-up,19.5,4,Yes,STF-04
69,TXN-10069,2026-03-26,CUST-0053,Pistachio Matcha,Drinks,1,6.5,Unknown,Card,Wholesale,6.5,2,Yes,STF-02
70,TXN-10069,2026-03-26,CUST-0053,Pistachio Matcha,Drinks,1,6.5,Unknown,Card,Wholesale,6.5,2,Yes,STF-02
145,TXN-10144,2026-06-23,CUST-0079,Matcha Starter Kit,Retail,2,47,Pop-up,Online,Wholesale,94.0,3,Yes,STF-03
146,TXN-10144,2026-06-23,CUST-0079,Matcha Starter Kit,Retail,2,47,Pop-up,Online,Wholesale,94.0,3,Yes,STF-03
204,TXN-10202,2026-09-04,Unknown,Pistachio Matcha,Drinks,5,6.5,Wholesale,Apple Pay,Wholesale,32.5,5,No,STF-03
205,TXN-10202,2026-09-04,Unknown,Pistachio Matcha,Drinks,5,6.5,Wholesale,Apple Pay,Wholesale,32.5,5,No,STF-03
233,TXN-10230,2026-09-30,CUST-0063,Pistachio Matcha,Drinks,1,6.5,Pop-up,Apple Pay,Mahon Pop-up,6.5,1,No,Unknown
234,TXN-10230,2026-09-30,CUST-0063,Pistachio Matcha,Drinks,1,6.5,Pop-up,Apple Pay,Mahon Pop-up,6.5,1,No,Unknown


In [17]:
#removing duplicated rows based on same transaction id but leving the first instance
sales = sales.drop_duplicates(subset=["Transaction_ID"], keep="first")

In [18]:
#check for duplicates again:
duplicates = sales.duplicated()
print(duplicates.sum())


0


In [19]:
#removing leading and trailing spaces from text values
#using function to remove only white spaces around string data type, leaving numbers intact as currently majority of colums are "object" data type

for col in sales.columns:
    sales[col] = sales[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

sales.head(10)

,Transaction_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price_EUR,Sales_Channel,Payment_Method,Location,Order_Total_EUR,Customer_Rating,Returned,Staff_ID
0,TXN-10001,2026-01-01,CUST-0052,Ceremonial Matcha 30g,Retail,1,24,Wholesale,Cash,Wholesale,24.0,4,Yes,STF-01
1,TXN-10002,2026-01-01,CUST-0056,Classic Matcha Latte,Drinks,1,5.5,In-store,Apple Pay,Mahon Pop-up,5.5,3,No,STF-03
2,TXN-10003,2026-01-01,CUST-0028,Iced Hojicha Latte,Drinks,3,5.8,In-store,Apple Pay,Mahon Pop-up,17.4,5,No,STF-01
3,TXN-10004,2026-01-03,CUST-0072,Iced Hojicha Latte,Drinks,2,5.8,Wholesale,Card,Marina Market,11.6,3,Yes,STF-04
4,TXN-10005,2026-01-08,CUST-0055,Matcha Granola,Food,3,7.5,Online,Apple Pay,Paul Street Pop-up,22.5,Unknown,No,STF-01
5,TXN-10006,2026-01-09,CUST-0039,Matcha Starter Kit,drink,1,47,Pop-up,Card,Mahon Pop-up,47.0,Unknown,Yes,STF-03
6,TXN-10007,2026-01-09,CUST-0012,Matcha Cookie,Food,1,3.7,In-store,Card,Marina Market,3.7,4,No,STF-01
7,TXN-10008,2026-01-10,CUST-0084,Iced Hojicha Latte,Drinks,1,5.8,Pop-up,Cash,Unknown,5.8,3,No,STF-02
8,TXN-10009,2026-01-12,CUST-0073,Pistachio Matcha,Drinks,two,6.5,Wholesale,Online,Online Store,13.0,Unknown,No,STF-02
9,TXN-10010,2026-01-12,CUST-0064,Ceremonial Matcha 30g,Retail,3,24,Pop-up,Online,Online Store,72.0,5,No,Unknown


In [20]:
# Checking for inconsistencies by inspecting unique values in categorical columns
for col in [ "Category",
    "Product",
    "Sales_Channel",
    "Payment_Method",
    "Returned",
    "Location",
    "Customer_Rating",
    "Staff_ID"]:
    print("\n")
    print(sales[col].value_counts(dropna=False))



Category
Drinks    123
Retail     66
Food       59
drink       1
RETAIL      1
Name: count, dtype: int64


Product
Matcha Starter Kit       41
Classic Matcha Latte     38
Pistachio Matcha         36
Matcha Cookie            35
Ceremonial Matcha 30g    27
Iced Hojicha Latte       27
Matcha Granola           23
Strawberry Matcha        23
Name: count, dtype: int64


Sales_Channel
Pop-up       69
Online       64
Wholesale    57
In-store     52
Unknown       5
in-store      1
POP UP        1
wholesale     1
Name: count, dtype: int64


Payment_Method
Apple Pay      67
Card           63
Online         59
Cash           54
Unknown         4
apple pay       1
Credit Card     1
online          1
Name: count, dtype: int64


Returned
No       191
Yes       54
N          1
Y          1
yes        1
NO         1
Maybe      1
Name: count, dtype: int64


Location
Wholesale             55
Mahon Pop-up          55
Paul Street Pop-up    47
Marina Market         44
Online Store          41
Unknown     

In [21]:
#Correct inconsistencies in columns with cathegorical values:

#Category
sales["Category"] = sales["Category"].replace({
    "drink": "Drinks",
    "RETAIL": "Retail"
})

#Sales_Channel
sales["Sales_Channel"] = sales["Sales_Channel"].replace({
    "in-store": "In-store",
    "POP UP": "Pop-up",
    "wholesale": "Wholesale"
})

#Payment_Method
sales["Payment_Method"] = sales["Payment_Method"].replace({
    "apple pay": "Apple Pay",
    "online": "Online",
    "Card": "Credit Card"
})

#Returned
sales["Returned"] = sales["Returned"].replace({
    "Y": "Yes",
    "N": "No",
    "yes": "Yes",
    "NO": "No"
})

#Location
sales["Location"] = sales["Location"].replace({
    "Marina market": "Marina Market",
    "Online store": "Online Store",
    "Paul street pop-up": "Paul Street Pop-up",
    "Mahon": "Mahon Pop-up"
})

#Customer_Rating
sales["Customer_Rating"] = sales["Customer_Rating"].replace({
    "4/5": 4,
    "five": 5
    })

In [22]:
# Check if inconsistencies were removed
for col in [ "Category",
    "Product",
    "Sales_Channel",
    "Payment_Method",
    "Returned",
    "Location",
    "Customer_Rating",
    "Staff_ID"]:
    print("\n")
    print(sales[col].value_counts(dropna=False))



Category
Drinks    124
Retail     67
Food       59
Name: count, dtype: int64


Product
Matcha Starter Kit       41
Classic Matcha Latte     38
Pistachio Matcha         36
Matcha Cookie            35
Ceremonial Matcha 30g    27
Iced Hojicha Latte       27
Matcha Granola           23
Strawberry Matcha        23
Name: count, dtype: int64


Sales_Channel
Pop-up       70
Online       64
Wholesale    58
In-store     53
Unknown       5
Name: count, dtype: int64


Payment_Method
Apple Pay      68
Credit Card    64
Online         60
Cash           54
Unknown         4
Name: count, dtype: int64


Returned
No       193
Yes       56
Maybe      1
Name: count, dtype: int64


Location
Mahon Pop-up          56
Wholesale             55
Paul Street Pop-up    48
Marina Market         45
Online Store          42
Unknown                4
Name: count, dtype: int64


Customer_Rating
Unknown    94
3          35
5          33
2          30
4          29
1          29
Name: count, dtype: int64


Staff_ID
STF-

In [23]:
#Checking value distributions and identifying potential inconsistencies in numerical columns

print(sales["Quantity"].value_counts(dropna=False))

print('\n')
print(sales["Unit_Price_EUR"].value_counts(dropna=False))

print('\n')
print(sales["Order_Total_EUR"].value_counts(dropna=False))

Quantity
1          114
2           66
3           33
4           17
5           13
10           3
two          1
-2           1
3 units      1
four         1
Name: count, dtype: int64


Unit_Price_EUR
47        41
6.5       36
5.5       35
3.7       35
5.8       27
24        26
7.5       23
6.3       23
€6.30      1
24 EUR     1
-5         1
5,5        1
Name: count, dtype: int64


Order_Total_EUR
3.7      19
5.8      17
6.5      15
24.0     14
94.0     14
47.0     14
5.5      13
7.4      12
6.3      12
11.0     10
7.5       9
13.0      8
16.5      7
12.6      7
15.0      6
19.5      6
11.6      6
188.0     6
48.0      5
72.0      5
22.5      4
141.0     4
11.1      4
32.5      4
235.0     3
26.0      3
22.0      3
37.5      2
17.4      2
55.0      2
25.2      2
18.9      2
96.0      2
31.5      1
29.0      1
75.0      1
23.2      1
5.0       1
120.0     1
27.5      1
30.0      1
Name: count, dtype: int64


In [24]:
#Correcting values:

#Quantity
sales["Quantity"] = sales["Quantity"].replace({
    -2: 2,
    "two": 2,
    "3 units": 3,
    "four": 4
})

#Unit_Price_EUR
sales["Unit_Price_EUR"] = sales["Unit_Price_EUR"].replace({
    "€6.30": 6.30,
    "24 EUR": 24,
    -5: 5,
    "5,5": 5.5
})


/var/folders/41/l2y869890g737crmyx8q_7t40000gn/T/ipykernel_16303/3767814513.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sales["Quantity"] = sales["Quantity"].replace({
/var/folders/41/l2y869890g737crmyx8q_7t40000gn/T/ipykernel_16303/3767814513.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sales["Unit_Price_EUR"] = sales["Unit_Price_EUR"].replace({


In [25]:
#Checking the results after correction,
print(sales["Quantity"].value_counts(dropna=False))

print('\n')
print(sales["Unit_Price_EUR"].value_counts(dropna=False))

print('\n')
print(sales["Order_Total_EUR"].value_counts(dropna=False))

Quantity
1     114
2      68
3      34
4      18
5      13
10      3
Name: count, dtype: int64


Unit_Price_EUR
47.0    41
5.5     36
6.5     36
3.7     35
24.0    27
5.8     27
6.3     24
7.5     23
5.0      1
Name: count, dtype: int64


Order_Total_EUR
3.7      19
5.8      17
6.5      15
24.0     14
94.0     14
47.0     14
5.5      13
7.4      12
6.3      12
11.0     10
7.5       9
13.0      8
16.5      7
12.6      7
15.0      6
19.5      6
11.6      6
188.0     6
48.0      5
72.0      5
22.5      4
141.0     4
11.1      4
32.5      4
235.0     3
26.0      3
22.0      3
37.5      2
17.4      2
55.0      2
25.2      2
18.9      2
96.0      2
31.5      1
29.0      1
75.0      1
23.2      1
5.0       1
120.0     1
27.5      1
30.0      1
Name: count, dtype: int64


In [26]:
#checking again the data types in the data set
sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 254
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Transaction_ID   250 non-null    object 
 1   Order_Date       250 non-null    object 
 2   Customer_ID      250 non-null    object 
 3   Product          250 non-null    object 
 4   Category         250 non-null    object 
 5   Quantity         250 non-null    int64  
 6   Unit_Price_EUR   250 non-null    float64
 7   Sales_Channel    250 non-null    object 
 8   Payment_Method   250 non-null    object 
 9   Location         250 non-null    object 
 10  Order_Total_EUR  250 non-null    float64
 11  Customer_Rating  250 non-null    object 
 12  Returned         250 non-null    object 
 13  Staff_ID         250 non-null    object 
dtypes: float64(2), int64(1), object(11)
memory usage: 29.3+ KB


In [27]:
#correcting data types
sales[["Staff_ID","Category","Sales_Channel","Payment_Method","Location","Returned","Product","Customer_Rating"]] = sales[["Staff_ID","Category","Sales_Channel","Payment_Method","Location","Returned","Product", "Customer_Rating"]].astype("category")
sales["Quantity"] = sales["Quantity"].astype("int64")
sales["Order_Date"] = sales["Order_Date"].astype("datetime64[ns]")

sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 254
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Transaction_ID   250 non-null    object        
 1   Order_Date       250 non-null    datetime64[ns]
 2   Customer_ID      250 non-null    object        
 3   Product          250 non-null    category      
 4   Category         250 non-null    category      
 5   Quantity         250 non-null    int64         
 6   Unit_Price_EUR   250 non-null    float64       
 7   Sales_Channel    250 non-null    category      
 8   Payment_Method   250 non-null    category      
 9   Location         250 non-null    category      
 10  Order_Total_EUR  250 non-null    float64       
 11  Customer_Rating  250 non-null    category      
 12  Returned         250 non-null    category      
 13  Staff_ID         250 non-null    category      
dtypes: category(8), datetime64[ns](1), float64(2), 

In [28]:
# Save cleaned dataset
OUTPUT_PATH = Path("data") / "retail_matcha_sales_cleaned.xlsx"

sales.to_excel(OUTPUT_PATH, index=False)

print(f"Cleaned dataset saved")

Cleaned dataset saved
